In [0]:
# /Volumes/workspace/default/food_inspections/chicago_inspection_dataset.csv
# /Volumes/workspace/default/food_inspections/dallas_inspection_dataset.csv
# Verify files exist
files = dbutils.fs.ls("/Volumes/workspace/default/food_inspections/")
for f in files:
    print(f.name, f.size)

In [0]:
# Read chicago CSV into a dataframe
chicago_raw = (
    spark.read
    .option("header", "true")
    .option("inferSchema", "true")
    .option("multiLine", "true")
    .option("escape", '"')
    .csv("/Volumes/workspace/default/food_inspections/chicago_inspection_dataset.csv")
)

print(f"Chicago rows: {chicago_raw.count()}, columns: {len(chicago_raw.columns)}")
chicago_raw.printSchema()
chicago_raw.show(5, truncate=False)

In [0]:
# Read Dallas CSV into a dataframe
dallas_raw = (
    spark.read
    .option("header", "true")
    .option("inferSchema", "true")
    .option("multiLine", "true")
    .option("escape", '"')
    .csv("/Volumes/workspace/default/food_inspections/dallas_inspection_dataset.csv")
)

print(f"Dallas rows: {dallas_raw.count()}, columns: {len(dallas_raw.columns)}")
dallas_raw.printSchema()
dallas_raw.show(5, truncate=False)

In [0]:
# Clean column names and write Bronze tables

import re

def clean_column_names(df):
    """Replace spaces, special chars with underscores for Delta compatibility"""
    for col_name in df.columns:
        new_name = re.sub(r'[^a-zA-Z0-9]', '_', col_name.strip())
        new_name = re.sub(r'_+', '_', new_name).strip('_').lower()
        df = df.withColumnRenamed(col_name, new_name)
    return df

# Clean and write Chicago
chicago_clean = clean_column_names(chicago_raw)
chicago_clean.write.mode("overwrite").option("overwriteSchema", "true").saveAsTable("workspace.default.bronze_chicago_inspections")

# Clean and write Dallas
dallas_clean = clean_column_names(dallas_raw)
dallas_clean.write.mode("overwrite").option("overwriteSchema", "true").saveAsTable("workspace.default.bronze_dallas_inspections")

print("Bronze tables created successfully!")

In [0]:
# Check new column names
chicago_bronze = spark.table("workspace.default.bronze_chicago_inspections")
dallas_bronze = spark.table("workspace.default.bronze_dallas_inspections")

print(f"Chicago Bronze: {chicago_bronze.count()} rows, {len(chicago_bronze.columns)} cols")
print(f"Dallas Bronze: {dallas_bronze.count()} rows, {len(dallas_bronze.columns)} cols")

print("\n--- Chicago columns ---")
for c in chicago_bronze.columns:
    print(f"  {c}")

print("\n--- Dallas columns (first 20) ---")
for c in dallas_bronze.columns[:20]:
    print(f"  {c}")